In [24]:
import joblib
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import MinMaxScaler

In [25]:
path = "gold_data.csv"
df = pd.read_csv(path)
df.head()

,Date,Close,Crude_Oil,Inflation,Interest_Rate,USD_EUR,USD_JPY
0,2019-01-02,1281.000000,59.41,252.561,2.4,1.1357,109.22
1,2019-01-03,1291.800049,59.41,252.561,2.4,1.1399,108.07
2,2019-01-04,1282.699951,59.41,252.561,2.4,1.1410,108.29
3,2019-01-07,1286.800049,59.41,252.561,2.4,1.1468,108.62
4,2019-01-08,1283.199951,59.41,252.561,2.4,1.1444,108.57


In [26]:
df = df.sort_values('Date').reset_index(drop=True)

In [27]:
feature_cols = ['Crude_Oil', 'Inflation', 'Interest_Rate', 'USD_EUR', 'USD_JPY']
target_col = ['Close']

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

df_features_scaled = pd.DataFrame(
    feature_scaler.fit_transform(df[feature_cols]),
    columns=feature_cols,
    index=df.index
)

df_target_scaled = pd.DataFrame(
    target_scaler.fit_transform(df[target_col]),
    columns=target_col,
    index=df.index
)

In [28]:
df_scaled = pd.concat([df['Date'], df_features_scaled, df_target_scaled], axis=1)
df_scaled.head()

,Date,Crude_Oil,Inflation,Interest_Rate,USD_EUR,USD_JPY,Close
0,2019-01-02,0.393271,0.0,0.445076,0.649869,0.113157,0.005412
1,2019-01-03,0.393271,0.0,0.445076,0.665547,0.093734,0.010408
2,2019-01-04,0.393271,0.0,0.445076,0.669653,0.097450,0.006198
3,2019-01-07,0.393271,0.0,0.445076,0.691303,0.103023,0.008095
4,2019-01-08,0.393271,0.0,0.445076,0.682344,0.102179,0.006429


In [29]:
def make_stationary(df, columns):
    for col in columns:
        p_value = adfuller(df[col])[1]
        if p_value > 0.05: 
            df[col] = df[col].diff()
    df = df.dropna()
    return df

In [17]:
df_scaled.replace([np.inf, -np.inf], np.nan, inplace=True)

df_scaled.fillna(method='ffill', inplace=True)

df_scaled.fillna(method='bfill', inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_4744\2216054026.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_scaled.fillna(method='ffill', inplace=True)
C:\Users\User\AppData\Local\Temp\ipykernel_4744\2216054026.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_scaled.fillna(method='bfill', inplace=True)


In [19]:
dataframe_stationary = make_stationary(df_scaled, df_scaled.columns.drop('Date')).copy()

date_series = (
    df['Date']
    .iloc[-dataframe_stationary.shape[0]:] 
    .reset_index(drop=True)
)
dataframe_stationary['Date'] = df['Date'].iloc[-dataframe_stationary.shape[0]:].reset_index(drop=True)

In [20]:
dataframe_stationary.head()

,Date,Crude_Oil,Inflation,Interest_Rate,USD_EUR,USD_JPY,Close
0,2019-01-02,0.0,0.0,0.0,0.015677,-0.019422,0.004996
1,2019-01-03,0.0,0.0,0.0,0.015677,-0.019422,0.004996
2,2019-01-04,0.0,0.0,0.0,0.004106,0.003716,-0.004209
3,2019-01-07,0.0,0.0,0.0,0.021650,0.005573,0.001897
4,2019-01-08,0.0,0.0,0.0,-0.008959,-0.000844,-0.001665


In [21]:
#--- now the dataset has stationary values. Mean is constant. Variance is constant. Autocorrelation structure is constant.

In [22]:
path = 'gold_data_scaled_stationary.csv'
dataframe_stationary.to_csv(path, index=False)

In [30]:
path = 'gold_data_scaled.csv'
df_scaled.to_csv(path, index=False)

In [11]:
joblib.dump( feature_scaler, 'scalers/feature_scaler.pkl')

['scalers/feature_scaler.pkl']

In [12]:
joblib.dump( target_scaler, 'scalers/target_scaler.pkl')

['scalers/target_scaler.pkl']